In [1]:
!pip install deep-translator chromadb sentence-transformers tqdm

In [2]:
import json
from tqdm import tqdm
from deep_translator import GoogleTranslator

In [3]:
with open("data's.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("✅ Total records:", len(data))

✅ Total records: 101


In [4]:
def chunk_text(text, size=500):
    return [text[i:i+size] for i in range(0, len(text), size)]

In [5]:
import re

def split_sentences(text):
    # Split by sentence endings (. ? !)
    sentences = re.split(r'(?<=[.!?]) +', text)
    return sentences

In [6]:
from deep_translator import GoogleTranslator
import time

def translate_to_tamil(text):
    sentences = split_sentences(text)
    tamil_sentences = []

    for sentence in sentences:
        success = False

        for attempt in range(3):  # 🔁 retry 3 times
            try:
                translated = GoogleTranslator(source='auto', target='ta').translate(sentence)
                tamil_sentences.append(translated)
                success = True
                break
            except Exception as e:
                print(f"⚠ Retry {attempt+1}:", e)
                time.sleep(2)

        if not success:
            tamil_sentences.append(sentence)  # fallback

    return " ".join(tamil_sentences)

In [7]:
tamil_data = []

for item in tqdm(data):
    tamil_text = translate_to_tamil(item["document"])

    tamil_data.append({
        "id": item["id"],
        "document": tamil_text,
        "metadata": item["metadata"]
    })

    time.sleep(1)  # 🚨 prevent blocking

 39%|███████████████████████████████▎                                                 | 39/101 [10:52<27:03, 26.18s/it]

⚠ Retry 1: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


 63%|███████████████████████████████████████████████████▎                             | 64/101 [18:10<15:02, 24.40s/it]

⚠ Retry 1: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


 93%|███████████████████████████████████████████████████████████████████████████▍     | 94/101 [24:41<01:32, 13.24s/it]

⚠ Retry 1: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


100%|████████████████████████████████████████████████████████████████████████████████| 101/101 [30:23<00:00, 18.05s/it]


In [8]:
with open("herbs_tamil.json", "w", encoding="utf-8") as f:
    json.dump(tamil_data, f, ensure_ascii=False, indent=2)

print("✅ Tamil dataset created successfully!")

✅ Tamil dataset created successfully!


In [9]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

In [10]:
collection_ta = client.create_collection(
    name="herbs_tamil",
    embedding_function=embedding_function
)

In [11]:
for item in tqdm(tamil_data):
    collection_ta.add(
        documents=[item["document"]],
        metadatas=[item["metadata"]],
        ids=[item["id"]]
    )

print("✅ Tamil Chroma DB created successfully!")

100%|████████████████████████████████████████████████████████████████████████████████| 101/101 [00:14<00:00,  6.90it/s]

✅ Tamil Chroma DB created successfully!


In [12]:
for item in tamil_data[:3]:
    print(item["document"][:200])

அபிஸ் பிண்ட்ரோ, பொதுவாக ஹிமாலயன் சில்வர் ஃபிர் அல்லது பிண்ட்ரோ ஃபிர் என அழைக்கப்படுகிறது, இது பினேசியே குடும்பத்தைச் சேர்ந்த ஒரு உயரமான பசுமையான ஊசியிலை மரமாகும், இது நேரான தண்டு, கூம்பு கிரீடம், வழுவ
அபீஸ் வெபியானா, பொதுவாக இந்திய சில்வர் ஃபிர், தாலிஸ்பத்ரா அல்லது ஹிமாலயன் ஃபிர் என அழைக்கப்படும், இது பினேசி குடும்பத்தைச் சேர்ந்த ஒரு உயரமான பசுமையான ஊசியிலை ஆகும், இது அதன் பிரமிடு வளர்ச்சிப் பழக்கம
ஆயுர்வேதத்தில் ரோசரி பட்டாணி, ஜெக்விரிட்டி அல்லது குஞ்சா என்று பொதுவாக அறியப்படும் அப்ரூஸ் ப்ரீகாடோரியஸ், ஃபேபேசியே குடும்பத்தைச் சேர்ந்த ஒரு மெல்லிய, வற்றாத, இரட்டை மலை ஏறுபவர் ஆகும், அதன் நுட்பமான ப
